# E04 — Rule-Based Classification Baseline (A0) — Analysis

**Research question**: How far can the existing fixed deterministic rule baseline go on NDA
classification before introducing LLM reasoning?

**Frozen A0 definition**: `pipeline/rule_baseline.py`, unmodified. Per-hypothesis (17 fixed
ContractNLI IDs) literal positive/negative keyword-phrase lists over full NDA document text.
Priority: any negative phrase match -> Contradiction; else any positive phrase match ->
Entailment; else -> NotMentioned (default). `classify_with_span` additionally returns the
matched phrase's character span as evidence; NotMentioned returns no evidence. No rule
ordering, phrase, or matching-semantics change was made for E04 — this experiment
*characterizes* the existing baseline, it does not optimize it.

**Historical exposure disclosure**: this exact implementation has previously been evaluated on
the full official DEV split (B02, `docs/experiments.md`, accuracy 59.9%/macro-F1 0.493) and the
full official TEST split (T041, 500-case and full 2,091-case runs) in the T-series
pre-reconstruction project. No reconstruction-v2 tuning uses those outcomes, and no rule
modification happens in E04. This TRAIN run is a reconstruction-v2 **characterization** of a
pre-existing baseline, not a claim that A0 was developed blind to DEV/TEST — it is not called
"blind" or "unseen" anywhere in this notebook.

**Input**: full NDA text, intentionally NOT routed through `retrieval_v1` — A0 is a separate,
cheaper architecture baseline (see `experiments/E04_rule_baseline/summary.md` section 3 for the
full justification). Later architecture comparison (E12) must not assume A0 and RAG share an
input-processing pipeline.

Local-only, $0, zero LLM/API calls.

In [1]:
import csv
import json
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parents[1] if Path.cwd().name == "E04_rule_baseline" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))
E04 = REPO_ROOT / "experiments/E04_rule_baseline"
RESULTS = E04 / "results"

summary = json.load(open(RESULTS / "run_E04_R0_train.json"))
rows = list(csv.DictReader(open(RESULTS / "rule_failure_analysis.csv")))
for r in rows:
    r["correct"] = r["correct"] == "True"
    r["rule_fired"] = r["rule_fired"] == "True"
print("n_cases:", summary["n_cases"])
print("verified_class_balance:", summary["verified_class_balance"])

n_cases: 7191
verified_class_balance: {'NotMentioned': 2820, 'Entailment': 3530, 'Contradiction': 841}


## 1-3. Research question, frozen A0 definition, historical exposure

Stated above. Verified programmatically below that the frozen rule module
(`pipeline/rule_baseline.py`) was not touched for this run — its `RULES` dict has exactly the
17 hypothesis IDs it has always had.

In [2]:
from pipeline.rule_baseline import RULES
print("hypothesis IDs with a defined rule:", len(RULES), sorted(RULES.keys()))
assert len(RULES) == 17

hypothesis IDs with a defined rule: 17 ['nda-1', 'nda-10', 'nda-11', 'nda-12', 'nda-13', 'nda-15', 'nda-16', 'nda-17', 'nda-18', 'nda-19', 'nda-2', 'nda-20', 'nda-3', 'nda-4', 'nda-5', 'nda-7', 'nda-8']


## 4. TRAIN universe — verified directly

In [3]:
assert summary["verified_class_balance"] == {"Entailment": 3530, "NotMentioned": 2820, "Contradiction": 841}
assert summary["n_cases"] == 7191
print("Verified: 423 documents x 17 hypotheses = 7,191 TRAIN cases, natural (unbalanced) distribution confirmed.")
print("No DEV, no TEST used anywhere in this E04 run.")

Verified: 423 documents x 17 hypotheses = 7,191 TRAIN cases, natural (unbalanced) distribution confirmed.
No DEV, no TEST used anywhere in this E04 run.


## 5-6. Classification metrics and confusion matrix

In [4]:
print(f"Accuracy:  {summary['accuracy']:.4f}")
print(f"Macro-F1:  {summary['macro_f1']:.4f}")
print()
print("Per-class recall:")
for label, r in summary["per_class_recall"].items():
    print(f"  {label:>13}: {r:.4f}")
print()
cm = summary["confusion_matrix"]
print(f"Confusion matrix (rows=gold, cols=predicted, order={cm['labels']}):")
for label, row in zip(cm["labels"], cm["matrix"]):
    print(f"  {label:>13}: {row}")

Accuracy:  0.5682
Macro-F1:  0.4709

Per-class recall:
     Entailment: 0.3799
  Contradiction: 0.1724
   NotMentioned: 0.9220

Confusion matrix (rows=gold, cols=predicted, order=['Entailment', 'Contradiction', 'NotMentioned']):
     Entailment: [1341, 170, 2019]
  Contradiction: [78, 145, 618]
   NotMentioned: [199, 21, 2600]


## 7. Contradiction Recall — shown separately, per the reconstruction brief's standing convention

In [5]:
lo, hi = summary["contradiction_recall_ci95"]
print(f"Contradiction Recall: {summary['contradiction_recall']:.4f} "
      f"(n={summary['contradiction_n']}, 95% CI [{lo:.4f}, {hi:.4f}])")

Contradiction Recall: 0.1724 (n=841, 95% CI [0.1484, 0.1994])


## 8. Rule-fire / default coverage — overall and by gold class

Important because a high NotMentioned score could come from defaulting rather than genuine
detection — this is checked directly, not assumed.

In [6]:
print("Overall rule-fire polarity:")
for k, v in summary["rule_polarity_counts"].items():
    print(f"  {k:>16}: {v:>5}  ({summary['rule_polarity_rate'][k]:.1%})")

print()
print("Coverage by gold class (does a rule fire, and correctly?):")
for label, counts in summary["coverage_by_gold_class"].items():
    total = sum(counts.values())
    print(f"  {label}: {counts}  (none_fired rate: {counts.get('none_fired', 0)/total:.1%})")

Overall rule-fire polarity:
        none_fired:  5237  (72.8%)
    positive_fired:  1618  (22.5%)
    negative_fired:   336  (4.7%)

Coverage by gold class (does a rule fire, and correctly?):
  Entailment: {'none_fired': 2019, 'positive_fired': 1341, 'negative_fired': 170}  (none_fired rate: 57.2%)
  Contradiction: {'none_fired': 618, 'positive_fired': 78, 'negative_fired': 145}  (none_fired rate: 73.5%)
  NotMentioned: {'none_fired': 2600, 'positive_fired': 199, 'negative_fired': 21}  (none_fired rate: 92.2%)


**Reading this table**: NotMentioned's strong recall (92.2%, section 6) is driven almost
entirely by the default fallback — 2,600 of 2,820 true-NotMentioned cases (92.2%) hit
`none_fired`, i.e. no rule matched anything, which is exactly correct behavior for that class.
By contrast, only 57.2% of Entailment cases (2,019/3,530 miss + 1,341/3,530 hit — see the
default-NotMentioned-overuse failure family below) and 73.5% of Contradiction cases (618/841
miss) trigger their intended rule at all. The rule baseline's real limitation is **coverage**,
not precision when it does fire (see section 9's evidence precision).

## 9-10. Evidence metrics and joint label+evidence correctness

Using the frozen E00 evidence-hit semantics (interval overlap between the rule's single
matched-phrase span and the document's annotated gold spans) and the frozen joint-metric rule
(`evaluation.metrics.joint_label_evidence_correctness`, tau=0.5): correct label alone is not
enough for Entailment/Contradiction — the matched span must overlap real gold evidence; for
NotMentioned, the label must be correct AND no evidence may be claimed. Full-document access is
never counted as evidence — only the explicit matched span.

In [7]:
print(f"Evidence-bearing cases (Entailment+Contradiction): {summary['evidence_bearing_n']}")
print(f"Evidence Recall:    {summary['evidence_recall']:.4f}")
print(f"Evidence Precision: {summary['evidence_precision']:.4f}  "
      "(of every span the rule claimed, including false-positive fires on true-NotMentioned "
      "documents, how often it was a real gold span)")
print()
print(f"Joint label+evidence correctness, overall: {summary['joint_label_evidence_correctness_overall']:.4f}")
print("Joint label+evidence correctness, by gold class:")
for label, rate in summary["joint_label_evidence_correctness_by_class"].items():
    print(f"  {label:>13}: {rate:.4f}")

Evidence-bearing cases (Entailment+Contradiction): 4371
Evidence Recall:    0.2899
Evidence Precision: 0.6484  (of every span the rule claimed, including false-positive fires on true-NotMentioned documents, how often it was a real gold span)

Joint label+evidence correctness, overall: 0.4823
Joint label+evidence correctness, by gold class:
     Entailment: 0.2184
  Contradiction: 0.1153
   NotMentioned: 0.9220


**Reading this**: evidence precision (64.8%) is much higher than evidence recall (29.0%) —
when the rule *does* claim a span, it is right about two-thirds of the time, but it frequently
either doesn't fire at all (coverage gap, section 8) or fires on the correct general topic
without landing inside the specific annotated span (see the "misaligned evidence" failure
family, section 12). NotMentoned's joint score (92.2%) equals its label recall exactly, since
correctly predicting NotMentioned with the rule's default (no-span) behavior automatically
satisfies the joint condition — this is a structural property of the metric for this
architecture, not a separate achievement.

## 11. Latency and cost

In [8]:
lat = summary["latency_ms"]
print(f"Mean latency:   {lat['mean']:.4f} ms/case")
print(f"Median latency: {lat['median']:.4f} ms/case")
print(f"P90 latency:    {lat['p90']:.4f} ms/case")
print(f"Total wall time for all {summary['n_cases']} cases: {summary['total_wall_seconds']:.2f} s")
print(f"Cost: ${summary['cost_usd']:.2f} -- no LLM/API calls were made")

Mean latency:   0.0731 ms/case
Median latency: 0.0518 ms/case
P90 latency:    0.1239 ms/case
Total wall time for all 7191 cases: 0.67 s
Cost: $0.00 -- no LLM/API calls were made


## 12. Failure analysis

Observed failure families, tagged only from actual case data (not assumed in advance —
E03's exception/carve-out finding on a different model is NOT assumed to be the main driver
here without checking).

In [9]:
import re
from collections import Counter

def collapse(family):
    if family.startswith("default NotMentioned overuse (Entailment"):
        return "default NotMentioned overuse (Entailment->NotMentioned, no phrase matched)"
    if family.startswith("default NotMentioned overuse (Contradiction"):
        return "default NotMentioned overuse (Contradiction->NotMentioned, no phrase matched)"
    if family.startswith("rule fired on distractor"):
        return "rule fired on distractor/false-positive text (true NotMentioned, wrongly labeled)"
    return family

collapsed = Counter(collapse(r["failure_family"]) for r in rows if not r["correct"])
for fam, n in collapsed.most_common():
    print(f"{n:>5}  {fam}")

print()
print("(", sum(collapsed.values()), "total errors,", len(rows) - sum(collapsed.values()), "correct )")

 2019  default NotMentioned overuse (Entailment->NotMentioned, no phrase matched)
  618  default NotMentioned overuse (Contradiction->NotMentioned, no phrase matched)
  220  rule fired on distractor/false-positive text (true NotMentioned, wrongly labeled)
  170  positive case matched a negative phrase (or vice versa) -- possible conflicting/ambiguous phrasing
   78  negative case matched a positive phrase -- possible conflicting/ambiguous phrasing

( 3105 total errors, 4086 correct )


**What the errors actually show** (not assumed in advance): the dominant failure mode by
far is **coverage**, not precision or a specific linguistic phenomenon like exception/carve-out
handling. 2,019 Entailment and 618 Contradiction cases simply never matched any phrase in their
hypothesis's keyword list and fell to the NotMentioned default — this is a vocabulary/paraphrase
coverage gap (the real NDA clause states the requirement in words the fixed keyword list
doesn't anticipate), not a negation or exception-handling problem specifically. A smaller but
real group (170 + 78 = 248 cases) reflects genuine positive/negative phrase conflicts — the
"wrong-direction" rule fired instead of the intended one, sometimes because a document contains
both phrasings (e.g. a general permission alongside a specific carve-out) and priority order
picked the wrong one. 220 cases are false positives on genuinely NotMentioned documents, where
a keyword phrase appeared in an unrelated context. Unlike E03's LLM-based analysis, this
data does **not** point primarily at exception/carve-out clauses — it points at plain lexical
coverage of the positive/negative phrase lists as the dominant limitation, which is the
expected failure mode for a literal-keyword-only system with no paraphrase or semantic
generalization at all.

## 13-14. Representative successes and failures

In [10]:
correct_examples = [r for r in rows if r["correct"]][:3]
print("--- Representative correct predictions ---")
for r in correct_examples:
    print(f"{r['case_id']}  gold={r['gold_label']}  pred={r['predicted_label']}  "
          f"matched_phrase={r['matched_phrase']!r}  evidence_hit={r['evidence_hit']}")

print()
print("--- Representative default-NotMentioned-overuse failures (Contradiction missed) ---")
c_misses = [r for r in rows if r["gold_label"] == "Contradiction" and not r["correct"]
            and not r["rule_fired"]][:3]
for r in c_misses:
    print(f"{r['case_id']}  gold={r['gold_label']}  pred={r['predicted_label']}  "
          f"rule_fired={r['rule_fired']}")

print()
print("--- Representative false-positive fires on true NotMentioned ---")
fp = [r for r in rows if "distractor" in r["failure_family"]][:3]
for r in fp:
    print(f"{r['case_id']}  gold={r['gold_label']}  pred={r['predicted_label']}  "
          f"matched_phrase={r['matched_phrase']!r}")

print()
print("--- Representative correct-label-misaligned-evidence cases ---")
misaligned = [r for r in rows if r["failure_family"] == "correct label, misaligned/wrong evidence span"][:3]
for r in misaligned:
    print(f"{r['case_id']}  gold={r['gold_label']}  pred={r['predicted_label']}  "
          f"matched_phrase={r['matched_phrase']!r}  evidence_hit={r['evidence_hit']}")

--- Representative correct predictions ---
train::34::nda-11  gold=NotMentioned  pred=NotMentioned  matched_phrase=''  evidence_hit=
train::34::nda-2  gold=NotMentioned  pred=NotMentioned  matched_phrase=''  evidence_hit=
train::34::nda-1  gold=Entailment  pred=Entailment  matched_phrase='designated as confidential'  evidence_hit=True

--- Representative default-NotMentioned-overuse failures (Contradiction missed) ---
train::86::nda-2  gold=Contradiction  pred=NotMentioned  rule_fired=False
train::86::nda-20  gold=Contradiction  pred=NotMentioned  rule_fired=False
train::86::nda-7  gold=Contradiction  pred=NotMentioned  rule_fired=False

--- Representative false-positive fires on true NotMentioned ---
train::86::nda-12  gold=NotMentioned  pred=Entailment  matched_phrase='independently develop'
train::87::nda-12  gold=NotMentioned  pred=Entailment  matched_phrase='independently develop'
train::88::nda-16  gold=NotMentioned  pred=Entailment  matched_phrase='shall return'

--- Representat

## 15. Comparison context with E03 — descriptive only, NOT an architecture verdict

E03 (`classification_prompt_v1` = P0, `qwen2.5:7b-instruct`) measured on **TRAIN_PROMPT_v1**,
a 150-case *balanced* diagnostic sample (50/50/50): accuracy 52.7%, Macro-F1 0.507,
Contradiction Recall 22.0%.

E04 (this experiment, A0 rule baseline) measured on the **full TRAIN split**, 7,191 cases,
*natural* (unbalanced) distribution: accuracy 56.8%, Macro-F1 0.471, Contradiction Recall
17.2%.

**These numbers are shown side by side for context only — they are NOT a valid architecture
comparison.** The populations differ (150 balanced cases vs. 7,191 natural-distribution cases):
E03's balanced sample deliberately equalizes NotMentoned's prevalence (33% of the sample) against
its true ~39% share of the natural distribution, and gives Contradiction 33% weight in the
sample vs. its true ~12% share — raw accuracy/macro-F1 are not comparable across differently-
distributed populations. The proper matched comparison (same cases, same metrics, all
architectures) is E12's job.

In [11]:
print(f"{'Metric':<24}{'E03 (Qwen P0, n=150 balanced)':<32}{'E04 (Rule A0, n=7191 natural)'}")
print(f"{'Accuracy':<24}{'52.7%':<32}{summary['accuracy']*100:.1f}%")
print(f"{'Macro-F1':<24}{'0.507':<32}{summary['macro_f1']:.3f}")
print(f"{'Contradiction Recall':<24}{'22.0%':<32}{summary['contradiction_recall']*100:.1f}%")

Metric                  E03 (Qwen P0, n=150 balanced)   E04 (Rule A0, n=7191 natural)
Accuracy                52.7%                           56.8%
Macro-F1                0.507                           0.471
Contradiction Recall    22.0%                           17.2%


## 16. Final A0 freeze

`A0_rule_baseline_v1` = `pipeline/rule_baseline.py`, unmodified from its T-series original.
Input: full NDA document text. Rule semantics: per-hypothesis literal positive/negative phrase
lists, negative-first priority, NotMentoned default. Evidence semantics: matched-phrase
character span, interval-overlapped against the document's annotated span list; no evidence
claimed for NotMentoned. TRAIN evaluation manifest: full official TRAIN split (423 docs x 17
hypotheses, 7,191 cases, natural distribution), verified directly. Historical DEV/TEST exposure
disclosed in full above and in `summary.md` — this is a characterization of a pre-existing,
unmodified baseline, not a blind or unseen result. No additional rule tuning was performed or
is proposed as part of this freeze.